# 02 — Fine-tune PaliGemma 2 with LoRA (PyTorch Lightning + Drive)

**Model:** `google/paligemma2-3b-pt-448`, 4-bit + LoRA.

**Storage strategy** (matches notebook 01):
- **JSONL files** read directly from Drive (tiny, fast).
- **Image zip** copied Drive → local `/content/` once per session and unzipped → fast reads during training.
- **Checkpoints + final adapter** saved to Drive so they survive disconnects and M3 can grab them.

## 0. Mount Drive, set paths, install

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from pathlib import Path
import os
import shutil

PROJECT_NAME = 'VU_DL_Team_Project'
DRIVE_ROOT   = Path('/content/drive/MyDrive') / PROJECT_NAME if IN_COLAB \
               else Path('./drive_local') / PROJECT_NAME

DATA_SUBDIR = 'pii_v1'
DATA_DRIVE   = DRIVE_ROOT / 'data' / DATA_SUBDIR
OUTPUTS      = DRIVE_ROOT / 'outputs'
ADAPTER_DIR  = OUTPUTS / 'lora_adapters' / 'final'
CKPT_DIR     = OUTPUTS / 'checkpoints'

# Local fast scratch for images
LOCAL_SCRATCH = Path('/content/scratch') if IN_COLAB else Path('./scratch')
DATA_LOCAL    = LOCAL_SCRATCH / DATA_SUBDIR

for p in [DATA_DRIVE, OUTPUTS, ADAPTER_DIR, CKPT_DIR, DATA_LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

# Required files
KEEP_FILES = ['train.jsonl', 'val.jsonl', 'test.jsonl', 'images.zip']

if IN_COLAB:
    # Check if all required files exist
    missing_files = [f for f in KEEP_FILES if not (DATA_DRIVE / f).exists()]

    if missing_files:
        print(f"Missing files: {missing_files}. Starting target-only download...")

        # 1. TELL THE CODE THE EXACT FILE IDs (Don't use the folder ID anymore)
        # To get these, right-click the file in Drive -> Share -> Copy Link.
        # The ID is the long string of letters and numbers in the middle of the link.
        FILE_IDS = {
            'train.jsonl': '1k4I7Dm_Zi52bR8gL2I4Vd1DEkAU1u_qT',
            'val.jsonl':   '1mrXN21YuQw-yWmNRrPosXvk2gB4sQZuS',
            'test.jsonl':  '16KRCMg8UUQS-YBuUkk1OMv-xMwcv0v2s',
            'images.zip':  '1tEkIGSHTAHjQgCdSSppRbpPh3IYki7Kv'
        }

        # 2. Loop through and download ONLY the missing files
        for file_name in missing_files:
            file_id = FILE_IDS.get(file_name)
            if file_id and not file_id.startswith('REPLACE_'):
                print(f"Downloading {file_name}...")
                destination = DATA_DRIVE / file_name

                # Download the single file directly
                !gdown {file_id} -O {str(destination)}
            else:
                print(f"⚠️ Skipping {file_name}: Valid File ID not provided yet.")

        print("✅ Target download check complete!")
    else:
        print("✅ All target files (train, val, test, images.zip) already exist in Drive. Skipping download.")

In [ ]:
!pip install -q \
  "transformers>=4.45" "accelerate>=0.34" "peft>=0.13" "bitsandbytes>=0.43" \
  "datasets" "pytorch-lightning>=2.4" "pillow" "tqdm"

## 1. Sync data to local disk (fast training reads)

In [ ]:
import shutil, zipfile, json, os, re, random
from PIL import Image
from tqdm.auto import tqdm

# Copy JSONL files (tiny)
for split in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    src = DATA_DRIVE / split
    dst = DATA_LOCAL / split
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f'Copied {split}')

# Unzip images to local disk (only once per session)
images_local = DATA_LOCAL / 'images'
if not images_local.exists() or not any(images_local.iterdir()):
    images_zip = DATA_DRIVE / 'images.zip'
    assert images_zip.exists(), f'Missing {images_zip} — re-run notebook 01 section 9'
    print(f'Unzipping {images_zip.stat().st_size/1e6:.1f} MB to local disk...')
    with zipfile.ZipFile(images_zip) as zf:
        zf.extractall(DATA_LOCAL)
    print(f'✅ {len(list(images_local.iterdir()))} images unzipped')
else:
    print(f'✅ images already on local disk: {len(list(images_local.iterdir()))}')

## 1b. Sanity check — object-count & label distribution
Confirms that `MAX_OBJECTS` is large enough and (if you flip `MULTI_CLASS=True` later) that label names match the taxonomy.

In [ ]:
import json, collections
for split in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    p = DATA_LOCAL / split
    if not p.exists():
        print(f'(skipping {split} — not found)'); continue
    rows = [json.loads(l) for l in open(p)]
    obj_counts = collections.Counter(len(r['objects']) for r in rows)
    labels = collections.Counter(o['label'] for r in rows for o in r['objects'])
    total_screens = sum(obj_counts.values())
    over_cap = sum(v for k, v in obj_counts.items() if k > 12)
    print(f'\n── {split}  ({total_screens} screens) ──')
    print('  objects/screen:', dict(sorted(obj_counts.items())))
    print(f'  screens with >12 boxes (over current MAX_OBJECTS): {over_cap}')
    print('  label distribution:', dict(labels.most_common()))

## 2. Config

In [ ]:
MODEL_ID    = 'google/paligemma2-3b-pt-448'
IMG_SIZE    = 448
BATCH_SIZE  = 1          # 448px PaliGemma = 1024 image tokens; bs=1 fits comfortably in ~12 GB
GRAD_ACCUM  = 8          # effective batch = 8
LR          = 2e-5
EPOCHS      = 8
MAX_OBJECTS = 12

MULTI_CLASS = False

MULTI_CLASS_LABELS = ['username',
                      'email_address',
                      'address',
                      'account_balance',
                      'date_of_birth',
                      'phone_number',
                      'full_name',
                      'transaction_amount']

if MULTI_CLASS:
    PROMPT = '<image>detect ' + ' ; '.join(MULTI_CLASS_LABELS) + '\n'
else:
    PROMPT = '<image>detect sensitive_info\n'
print('PROMPT =', repr(PROMPT))

# ── LoRA scope ─────────────────────────────────────────────────────────────────
LORA_R       = 32                                   # up from 16
LORA_ALPHA   = 64                                   # 2× rank, standard convention
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj',
                'gate_proj','up_proj','down_proj']  # added MLP modules
UNFREEZE_PROJECTOR = False

# ── Generation cap (inference only) ────────────────────────────────────────────
# Each box = 4 loc tokens + label + separator ≈ 10 tokens.
# 192 tokens ≈ up to ~18 boxes per screen — well above MAX_OBJECTS.
GEN_MAX_TOKENS = 192
HF_TOKEN    = os.environ.get('HF_TOKEN', None)

import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping
from transformers import (PaliGemmaProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig, get_cosine_schedule_with_warmup)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.set_float32_matmul_precision('medium')
pl.seed_everything(42)

PROMPT = '<image>detect sensitive_info\n'


INFO:lightning_fabric.utilities.seed:Seed set to 42


42

## 3. PaliGemma bbox encoding (loc tokens, Y-first, 0–1023)

In [ ]:
import re
from typing import List, Dict, Union


def pixels_to_loc_tokens(bbox_xyxy_px: List[int], img_w: int, img_h: int) -> str:
    """
    Converts pixel coordinates to PaliGemma <locxxxx> tokens.

    Args:
        bbox_xyxy_px: A list of [x1, y1, x2, y2] in pixels.
        img_w: Width of the image.
        img_h: Height of the image.

    Returns:
        A string containing four <loc> tokens.
    """
    x1, y1, x2, y2 = bbox_xyxy_px

    def n(v: float, d: int) -> int:
        return max(0, min(1023, int(round(v / d * 1024))))

    return (f'<loc{n(y1, img_h):04d}><loc{n(x1, img_w):04d}>'
            f'<loc{n(y2, img_h):04d}><loc{n(x1, img_w):04d}>')


LOC_RE = re.compile(r'<loc(\d{4})><loc(\d{4})><loc(\d{4})><loc(\d{4})>\s*([A-Za-z_][\w_]*)?')


def loc_tokens_to_pixels(text, img_w, img_h):
    out = []
    for m in LOC_RE.finditer(text):
        ny1, nx1, ny2, nx2, label = m.groups()

        # 1. Convert to raw pixels
        y1, x1 = int(ny1)/1024*img_h, int(nx1)/1024*img_w
        y2, x2 = int(ny2)/1024*img_h, int(nx2)/1024*img_w

        # 2. Safety: Ensure x1 is always the left side, y1 is the top
        x1, x2 = min(x1, x2), max(x1, x2)
        y1, y2 = min(y1, y2), max(y1, y2)

        # 3. Safety: Force a minimum 1-pixel width/height to prevent OpenCV crashes
        if x2 == x1: x2 += 1
        if y2 == y1: y2 += 1

        out.append({
            'bbox': [int(x1), int(y1), int(x2), int(y2)],
            'label': label or 'sensitive_info'
        })
    return out

## 4. Dataset (paths resolved against local scratch)

In [ ]:
from pathlib import Path
from typing import Any, Dict, List


class ScreenBboxDataset(Dataset):
    """
    Dataset for ScreenQA PII detection using PaliGemma.
    """

    def __init__(
        self,
        jsonl_path: Union[str, Path],
        data_dir: Union[str, Path],
        prompt: str = PROMPT,
        max_objects: int = MAX_OBJECTS
    ):
        self.data_dir = Path(data_dir)
        self.prompt = prompt
        self.max_objects = max_objects
        with open(jsonl_path, 'r') as f:
            self.rows = [json.loads(line) for line in f]

    def __len__(self) -> int:
        return len(self.rows)

    def _target(self, objects: List[Dict[str, Any]], w: int, h: int) -> str:
        if not objects:
            return 'none'

        def label_for(obj: Dict[str, Any]) -> str:
            return obj['label'] if MULTI_CLASS else 'sensitive_info'

        parts = [
            f"{pixels_to_loc_tokens(o['bbox'], w, h)} {label_for(o)}"
            for o in objects[:self.max_objects]
        ]
        return ' ; '.join(parts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.rows[idx]
        img = Image.open(self.data_dir / ex['image']).convert('RGB')
        return {
            'image': img,
            'prompt': self.prompt,
            'target': self._target(ex['objects'], ex['image_width'], ex['image_height'])
        }


processor = PaliGemmaProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)


def collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    """
    Collate function to process batch through PaliGemmaProcessor.
    """
    return processor(
        text=[b['prompt'] for b in batch],
        images=[b['image'] for b in batch],
        suffix=[b['target'] for b in batch],
        return_tensors='pt',
        padding='longest'
    )

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


## 5. Lightning DataModule + Model

In [ ]:
class DM(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=BATCH_SIZE, num_workers=2):
        super().__init__()
        self.data_dir, self.bs, self.nw = Path(data_dir), batch_size, num_workers
    def setup(self, stage=None):
        self.tr = ScreenBboxDataset(self.data_dir / 'train.jsonl', self.data_dir)
        self.va = ScreenBboxDataset(self.data_dir / 'val.jsonl',   self.data_dir)
    def train_dataloader(self):
        return DataLoader(self.tr, batch_size=self.bs, shuffle=True,
                          num_workers=self.nw, collate_fn=collate)
    def val_dataloader(self):
        return DataLoader(self.va, batch_size=self.bs, shuffle=False,
                          num_workers=self.nw, collate_fn=collate)

In [ ]:
class PaliGemmaLora(pl.LightningModule):
    """
    PyTorch Lightning Module for fine-tuning the PaliGemma 2 Vision-Language Model.

    This module utilizes 4-bit NormalFloat (NF4) quantization via BitsAndBytes and
    Low-Rank Adaptation (LoRA) via PEFT to enable efficient training on consumer GPUs.
    It automatically freezes the vision encoder and applies a cosine learning rate
    scheduler with warmup.

    Args:
        model_id (str): Hugging Face model identifier (e.g., 'google/paligemma2-3b-pt-448').
        lr (float): Maximum learning rate for the AdamW optimizer.
    """

    def __init__(self, model_id: str = MODEL_ID, lr: float = LR) -> None:
        super().__init__()
        self.save_hyperparameters()

        # 1. Configure 4-bit Quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

        # 2. Load Base Model
        base_model = PaliGemmaForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            torch_dtype=torch.bfloat16,
            token=HF_TOKEN,
        )

        # 3. Prepare for k-bit training (enables gradient checkpointing for memory efficiency)
        base_model = prepare_model_for_kbit_training(
            base_model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

        # 4. Freeze the Vision Tower to prevent catastrophic forgetting
        for name, param in base_model.named_parameters():
            if "vision_tower" in name:
                param.requires_grad = False

        # 4b. Optionally unfreeze the multimodal projector
        if UNFREEZE_PROJECTOR:
            unfrozen = 0
            for name, param in base_model.named_parameters():
                if 'multi_modal_projector' in name:
                    # Cast to bf16 first if it's quantized (int dtype can't take grads)
                    if param.dtype in (torch.uint8, torch.int8):
                        param.data = param.data.to(torch.bfloat16)
                    param.requires_grad = True
                    unfrozen += 1
            print(f"Unfroze {unfrozen} projector params")
            assert unfrozen > 0, "No multi_modal_projector params found — check module path"

        # 5. Configure and Apply LoRA Adapters (language model only)
        LANG_SUFFIXES = ('q_proj', 'k_proj', 'v_proj', 'o_proj',
                         'gate_proj', 'up_proj', 'down_proj')
        # Match only modules inside the language model; explicitly exclude vision_tower
        target_module_names = [
            name for name, _ in base_model.named_modules()
            if name.endswith(LANG_SUFFIXES)
            and 'language_model' in name
            and 'vision_tower' not in name
        ]
        assert target_module_names, "No LoRA targets found — check module naming."
        assert not any('vision_tower' in n for n in target_module_names), \
            "vision_tower leaked into LoRA targets!"
        print(f"LoRA will be applied to {len(target_module_names)} language-model modules")

        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=target_module_names,   # full paths, not suffixes
            task_type="CAUSAL_LM",
        )
        self.model = get_peft_model(base_model, lora_config)
        self.model.print_trainable_parameters()

        # 6. Sanity Check: nothing trainable should live inside vision_tower
        vt_trainable = [
            n for n, p in self.model.named_parameters()
            if p.requires_grad and 'vision_tower' in n
        ]
        if vt_trainable:
            raise RuntimeError(
                f"{len(vt_trainable)} vision_tower params are trainable! "
                f"First: {vt_trainable[0]}"
            )

        extra_trainable = [
            n for n, p in self.model.named_parameters()
            if p.requires_grad and 'lora_' not in n
        ]
        example_str = f"  (e.g., {extra_trainable[0]})" if extra_trainable else ""
        print(f"Non-LoRA trainable tensors: {len(extra_trainable)}{example_str}")

    def training_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        """
        Processes a single training batch and calculates the loss.
        """
        outputs = self.model(**batch)
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        """
        Processes a single validation batch and calculates the loss.
        """
        outputs = self.model(**batch)
        loss = outputs.loss
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def configure_optimizers(self) -> Dict[str, Any]:
        """
        Configures the AdamW optimizer and the Cosine Annealing scheduler with warmup.

        Returns:
            Dict containing the optimizer and the learning rate scheduler configuration.
        """
        # Collect only the unfrozen, trainable parameters (LoRA weights)
        trainable_params = [p for p in self.parameters() if p.requires_grad]

        optimizer = torch.optim.AdamW(
            trainable_params,
            lr=self.hparams.lr,
            weight_decay=1e-2,
        )

        # Calculate steps for the scheduler dynamically based on the Lightning Trainer
        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = int(total_steps * 0.1)  # 10% Warmup period

        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",  # Evaluate the scheduler per-batch, not per-epoch
                "frequency": 1,
            },
        }

## 6. Train (checkpoints → Drive, auto-resume on disconnect)

In [ ]:
dm    = DM(DATA_LOCAL)
model = PaliGemmaLora()

ckpt_cb = ModelCheckpoint(
    dirpath=str(CKPT_DIR), monitor='val_loss', mode='min',
    save_top_k=2, save_last=True,
    filename='pg2-{epoch:02d}-{val_loss:.4f}',
)
early_stop_cb = EarlyStopping(
    monitor='val_loss', mode='min', patience=4, verbose=True,
)

resume_from = CKPT_DIR / 'last.ckpt'
resume_arg = str(resume_from) if resume_from.exists() else None
if resume_arg:
    print(f'🔁 Resuming from {resume_arg}')

trainer = pl.Trainer(
    max_epochs=EPOCHS, accelerator='gpu', devices=1,
    precision='bf16-mixed',
    accumulate_grad_batches=GRAD_ACCUM, gradient_clip_val=1.0,
    callbacks=[ckpt_cb, early_stop_cb, LearningRateMonitor(logging_interval='step')],
    log_every_n_steps=10, val_check_interval=0.5,
)
trainer.fit(model, datamodule=dm, ckpt_path=resume_arg)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

LoRA will be applied to 182 language-model modules


INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


trainable params: 41,533,440 || all params: 3,074,660,592 || trainable%: 1.3508
Non-LoRA trainable tensors: 0


...

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ PeftModelForCausalLM │  1.9 B │ train │     0 │
└───┴───────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 41.5 M                                                                                           
Non-trainable params: 1.8 B                                                                                        
Total params: 1.9 B                                                                                                
Total estimated model params size (MB): 7,422.641                                                                  
Modules in train mode: 1822                                                                                        
Modules in eval mode: 732                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:538: Found 732 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved. New best score: 1.758
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.037 >= min_delta = 0.0. New best score: 1.721
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.243 >= min_delta = 0.0. New best score: 1.478
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.013 >= min_delta = 0.0. New best score: 1.465
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.034 >= min_delta = 0.0. New best score: 1.432
INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.028 >= min_delta = 0.0. New best score: 1.404
INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 4 records. Best score: 1.404. Signaling Trainer to stop.


## 7. Save final LoRA adapter to Drive (for M3)

In [ ]:
import torch
import gc

# 1. Clear memory to create as much headroom as possible
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

# 2. Path check
CERTAIN_CKPT_PATH = CKPT_DIR / 'pg2-epoch=02-val_loss=1.4039.ckpt'
if not CERTAIN_CKPT_PATH.exists():
    print(f"❌ Error: Checkpoint not found at {CERTAIN_CKPT_PATH}")
else:
    print(f"Loading checkpoint (memory-optimized): {CERTAIN_CKPT_PATH}")
    try:
        # Ensure we define 'model' globally so other cells can see it
        global model

        # 3. Load using low_cpu_mem_usage to prevent the silent OOM crash
        # This is critical for 3B+ parameter models in Colab RAM
        model = PaliGemmaLora.load_from_checkpoint(
            str(CERTAIN_CKPT_PATH),
            map_location='cpu',
            strict=False,
            low_cpu_mem_usage=True
        )

        # Move to GPU safely after initial load
        model.to('cuda' if torch.cuda.is_available() else 'cpu')
        model.eval()
        print("✅ Loaded ✓")

        # 4. Save components to Drive
        ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
        model.model.save_pretrained(str(ADAPTER_DIR))
        processor.save_pretrained(str(ADAPTER_DIR))
        print(f'💾 Saved adapter + processor to {ADAPTER_DIR}')
    except Exception as e:
        print(f"❌ Failed to load model: {e}")

Loading checkpoint (memory-optimized): /content/drive/MyDrive/VU_DL_Team_Project/outputs/checkpoints/pg2-epoch=02-val_loss=1.4039.ckpt


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

LoRA will be applied to 182 language-model modules
trainable params: 41,533,440 || all params: 3,074,660,592 || trainable%: 1.3508
Non-LoRA trainable tensors: 0


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.weight.absmax', 'model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.weight.quant_map', 'model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.weight.nested_absmax', 'model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.weight.nested_quant_map', 'model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.weight.quant_state.bitsandbytes__nf4', 'model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.weight.absmax', 'model.base_model.model.model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.weight.quant_map', 'model.base_model.model.model.vision_tower.vision_model.encode

✅ Loaded ✓
💾 Saved adapter + processor to /content/drive/MyDrive/VU_DL_Team_Project/outputs/lora_adapters/final


## 8. Quick inference sanity check

In [ ]:
import time
from peft import PeftModel

# 1. Define 4-bit config
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

# 2. Load base model
base = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.bfloat16,
    device_map='auto', token=HF_TOKEN,
)

# 3. Load the best adapter weights saved during training
# Using ADAPTER_DIR which contains the exported best weights from cell gZJ3SYH0dr3E
infer_model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
infer_model.eval()
infer_model.config.use_cache = True

if hasattr(infer_model, 'gradient_checkpointing_disable'):
    infer_model.gradient_checkpointing_disable()

print('Inference model device:', next(infer_model.parameters()).device)
print(f'Loaded best adapter from: {ADAPTER_DIR}')

# 4. Run a random validation example
val_examples = [json.loads(l) for l in open(DATA_LOCAL / 'val.jsonl')]
ex = random.choice([e for e in val_examples if e['objects']])
img = Image.open(DATA_LOCAL / ex['image']).convert('RGB')

inference_task = PROMPT.rstrip('\n')
inputs = processor(text=inference_task, images=img, return_tensors='pt').to(infer_model.device)

t0 = time.time()
with torch.inference_mode():
    out = infer_model.generate(**inputs, max_new_tokens=GEN_MAX_TOKENS,
                               num_beams=3, do_sample=False, use_cache=True)

print(f'Generation took: {time.time()-t0:.2f}s')
decoded = processor.batch_decode(out, skip_special_tokens=False)[0]
print('RAW :', decoded[-300:])
print('PRED:', loc_tokens_to_pixels(decoded, ex['image_width'], ex['image_height']))
print('GT  :', ex['objects'])

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

Inference model device: cuda:0
Loaded best adapter from: /content/drive/MyDrive/VU_DL_Team_Project/outputs/lora_adapters/final
Generation took: 0.39s
RAW : <image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><bos>detect sensitive_info
<loc0807><loc0119><loc0836><loc0119> sensitive_info<eos>
PRED: [{'bbox': [125, 1513, 126, 1567], 'label': 'sensitive_info'}]
GT  : [{'bbox': [118, 783, 597, 837], 'label': 'email_address', 'source_question': 'What is the shown email address?', 'source_answer': 'The shown email address is WICShopper@jpma.com.'}]
